# Week 3: 재생성 파이프라인 + Exp 2, 3

**목표**
1. 3단계 재생성 전략(Baseline/MID-A/HIGH/Ours) 비교 (Exp 2)
2. MID-A(top-p) vs MID-B(DoLa) 상세 비교 (Exp 3)

**실행 환경**: Google Colab Pro (A100 40GB)

**예상 시간**: A100 기준 약 30~50분

> exp1_probing_result.pkl 이 Drive hs_cache에 있어야 합니다 (Week 2 완료 필요)

## 0. 환경 세팅

In [ ]:
!nvidia-smi | head -5

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/self-correcting-llm'
HS_CACHE  = f'{DRIVE_DIR}/hs_cache'
os.environ['HS_CACHE'] = HS_CACHE

# HaluEval을 Drive에서 로컬로 복사
import shutil
from pathlib import Path

local_halu = 'data/halueval_qa.json'
os.makedirs('data', exist_ok=True)
if not Path(local_halu).exists():
    shutil.copy(f'{DRIVE_DIR}/halueval_qa.json', local_halu)
    print('HaluEval 복사 완료')
else:
    print('HaluEval 이미 존재')

# Drive 파일 확인
pkl_ok = Path(HS_CACHE + '/exp1_probing_result.pkl').exists()
print(f'exp1_probing_result.pkl : {"OK" if pkl_ok else "없음 — Week 2 먼저 실행 필요"}')

In [ ]:
import os
REPO = 'self-correcting-llm'
if os.path.exists('config.py'):
    print(f'레포 루트: {os.getcwd()}')
elif os.path.exists(REPO):
    %cd {REPO}
else:
    !git clone -b phase/3-regen-pipeline-exp2-3 \
        https://github.com/uuyeong/self-correcting-llm.git
    %cd {REPO}

In [ ]:
!pip install -q -r requirements.txt
print('설치 완료')

In [ ]:
import sys
sys.path.insert(0, '.')
import config

config.HALUEVAL_PATH = 'data/halueval_qa.json'
print(f'BEST_LAYER    : {config.BEST_LAYER}')
print(f'HALUEVAL_PATH : {config.HALUEVAL_PATH}')
print(f'HS_CACHE      : {os.environ["HS_CACHE"]}')

---
## 1. Exp 2: 3단계 전략 비교

- **Baseline**: 항상 pass-through (hallucinated answer 그대로 반환)
- **MID-A**: 항상 top-p=0.7로 재생성
- **HIGH**: 항상 full 재생성
- **Ours**: 분류기 판단에 따라 LOW/MID-A/HIGH 선택

200개 HaluEval 질문 쌍 사용, 예상 시간: ~25분

In [ ]:
%run experiments/exp2_strategy_comparison.py

In [ ]:
import json, matplotlib.pyplot as plt
from src.visualization.plots import plot_strategy_comparison

with open('results/exp2_strategy_results.json') as f:
    data = json.load(f)

print('Exp 2 결과:')
for name in ['Baseline', 'MID-A', 'HIGH', 'Ours']:
    m = data[name]
    print(f'  {name:8s}: acc={m["accuracy"]:.3f}, cost={m["token_cost"]:.1f} tok, lat={m["latency_ms"]:.1f} ms')
print(f'\nLevel 분포: {data["_meta"]["level_counts"]}')

fig = plot_strategy_comparison({k: data[k] for k in ['Baseline', 'MID-A', 'HIGH', 'Ours']}, save=False)
plt.show()

---
## 2. Exp 3: MID-A vs MID-B

분류기가 MID로 판정한 샘플에 대해서만 두 전략 비교

- **MID-A**: top-p=0.7 샘플링
- **MID-B**: DoLa (layer 16 vs layer 8 logit 대조)

예상 시간: ~15분 (MID-B DoLa가 느림)

In [ ]:
%run experiments/exp3_mid_strategy.py

In [ ]:
import json, matplotlib.pyplot as plt
from src.visualization.plots import plot_mid_ab_comparison

with open('results/exp3_mid_comparison.json') as f:
    data = json.load(f)

print('Exp 3 결과:')
print(f'  MID-A: acc={data["MID-A"]["accuracy"]:.3f}, lat={data["MID-A"]["latency_ms"]:.1f} ms')
print(f'  MID-B: acc={data["MID-B"]["accuracy"]:.3f}, lat={data["MID-B"]["latency_ms"]:.1f} ms')
print(f'  MID 샘플 수: {data["_meta"]["n_mid_samples"]}')

fig = plot_mid_ab_comparison(data['MID-A'], data['MID-B'], save=False)
plt.show()

---
## 3. Week 3 완료 체크리스트

In [ ]:
from pathlib import Path

checks = [
    ('exp2_strategy_results.json',             Path('results/exp2_strategy_results.json').exists()),
    ('exp2_strategy_comparison.pdf',           Path('results/figures/exp2_strategy_comparison.pdf').exists()),
    ('exp3_mid_comparison.json',               Path('results/exp3_mid_comparison.json').exists()),
    ('exp3_mid_ab.pdf',                        Path('results/figures/exp3_mid_ab.pdf').exists()),
    ('exp2_hidden_states.npy (Drive)',         Path(HS_CACHE + '/exp2_hidden_states.npy').exists()),
    ('exp3_hidden_states.npy (Drive)',         Path(HS_CACHE + '/exp3_hidden_states.npy').exists()),
]

all_pass = True
for name, ok in checks:
    mark = 'v' if ok else 'x'
    print(f'  [{mark}] {name}')
    if not ok:
        all_pass = False

print()
print('Week 3 완료!' if all_pass else '미완료 항목 있음')

---
## 4. Git 커밋 & 푸시

```bash
git add experiments/exp2_strategy_comparison.py \
        experiments/exp3_mid_strategy.py \
        results/exp2_strategy_results.json \
        results/exp3_mid_comparison.json \
        results/figures/exp2_strategy_comparison.pdf \
        results/figures/exp3_mid_ab.pdf \
        notebooks/week3_regen_pipeline_exp2_3.ipynb
git commit -m "feat(exp2,3): 재생성 전략 비교 및 MID-A/B 분석 완료"
git push origin phase/3-regen-pipeline-exp2-3
```